In [42]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset


class SAR_Dataset(Dataset):
    """Single-modal dataset for Sentinel-1 SAR (grayscale) images."""

    def __init__(self, base_dir, transform=None):
        self.transform = transform
        self.data = self._build_data(base_dir)

    def _build_data(self, base_dir):
        classes = {"agri": 0, "barrenland": 1, "grassland": 2, "urban": 3}
        data = []
        for cls_name, label in classes.items():
            sar_dir = os.path.join(base_dir, cls_name, "s1")
            for fname in sorted(os.listdir(sar_dir)):
                if fname.endswith(".png"):
                    data.append((os.path.join(sar_dir, fname), label))
        return data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert("L")
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)


class Optical_Dataset(Dataset):
    """Single-modal dataset for Sentinel-2 Optical (RGB) images."""

    def __init__(self, base_dir, transform=None):
        self.transform = transform
        self.data = self._build_data(base_dir)

    def _build_data(self, base_dir):
        classes = {"agri": 0, "barrenland": 1, "grassland": 2, "urban": 3}
        data = []
        for cls_name, label in classes.items():
            opt_dir = os.path.join(base_dir, cls_name, "s2")
            for fname in sorted(os.listdir(opt_dir)):
                if fname.endswith(".png"):
                    data.append((os.path.join(opt_dir, fname), label))
        return data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)


class Fusion_Dataset(Dataset):
    """
    Dual-modal dataset that returns (sar_tensor, optical_tensor, label) triplets.

    Pairing logic:
        SAR    path: .../cls/s1/ROIs1868_summer_s1_59_p100.png
        Optical path: .../cls/s2/ROIs1868_summer_s2_59_p100.png

    The optical filename is derived from the SAR filename by replacing
    the '_s1_' token with '_s2_' in the filename only (directory s1→s2).
    Pairs where the expected optical file does not exist are skipped with
    a warning so training still proceeds cleanly.
    """

    def __init__(self, base_dir, sar_transform=None, optical_transform=None):
        self.sar_transform = sar_transform
        self.optical_transform = optical_transform
        self.pairs = self._build_pairs(base_dir)

    def _build_pairs(self, base_dir):
        classes = {"agri": 0, "barrenland": 1, "grassland": 2, "urban": 3}
        pairs = []
        skipped = 0
        for cls_name, label in classes.items():
            sar_dir = os.path.join(base_dir, cls_name, "s1")
            opt_dir = os.path.join(base_dir, cls_name, "s2")
            for fname in sorted(os.listdir(sar_dir)):
                if not fname.endswith(".png"):
                    continue
                opt_fname = fname.replace("_s1_", "_s2_")
                sar_path = os.path.join(sar_dir, fname)
                opt_path = os.path.join(opt_dir, opt_fname)
                if not os.path.exists(opt_path):
                    print(f"[Fusion_Dataset] WARNING: No optical pair for {sar_path}. Skipping.")
                    skipped += 1
                    continue
                pairs.append((sar_path, opt_path, label))
        print(f"[Fusion_Dataset] Total valid pairs: {len(pairs)}  |  Skipped: {skipped}")
        return pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        sar_path, opt_path, label = self.pairs[idx]
        sar_img = Image.open(sar_path).convert("L")
        opt_img = Image.open(opt_path).convert("RGB")
        if self.sar_transform:
            sar_img = self.sar_transform(sar_img)
        if self.optical_transform:
            opt_img = self.optical_transform(opt_img)
        return sar_img, opt_img, torch.tensor(label, dtype=torch.long)

In [43]:
sar_model = SAR_Dataset("/kaggle/input/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain/v_2")
print(len(sar_model))

      
optical_model = Optical_Dataset("/kaggle/input/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain/v_2")
print(len(optical_model))

      
fusion_model=Fusion_Dataset("/kaggle/input/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain/v_2")

print(len(fusion_model))


16000
16000
[Fusion_Dataset] Total valid pairs: 16000  |  Skipped: 0
16000


In [44]:
#model.py
import torch
import torch.nn as nn


# ── Shared building blocks ─────────────────────────────────────────────────────

def _make_conv_block(in_channels, out_channels):
    """Conv2d(3×3, pad=1) → BatchNorm → ReLU → MaxPool(2×2)."""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )


def _build_feature_extractor(in_channels):
    """
    5 conv blocks:  in_channels → 32 → 64 → 64 → 128 → 128
    Input 256×256   →  output shape  128 × 8 × 8
    Flattened dim   =  128 × 8 × 8  =  8 192
    """
    return nn.Sequential(
        _make_conv_block(in_channels, 32),
        _make_conv_block(32, 64),
        _make_conv_block(64, 64),
        _make_conv_block(64, 128),
        _make_conv_block(128, 128),
    )


FLAT_DIM = 128 * 8 * 8   # 8192  — single-branch flattened size


# ── Single-modal models ────────────────────────────────────────────────────────

class SAR_model(nn.Module):
    """Single-modal CNN for Sentinel-1 SAR (1-channel grayscale)."""

    def __init__(self, num_classes=4):
        super().__init__()
        self.features = _build_feature_extractor(in_channels=1)
        self.classifier = nn.Sequential(
            nn.Linear(FLAT_DIM, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


class Optical_model(nn.Module):
    """Single-modal CNN for Sentinel-2 Optical (3-channel RGB)."""

    def __init__(self, num_classes=4):
        super().__init__()
        self.features = _build_feature_extractor(in_channels=3)
        self.classifier = nn.Sequential(
            nn.Linear(FLAT_DIM, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


# ── Late-fusion model ──────────────────────────────────────────────────────────

class Fusion_model(nn.Module):
    """
    Late-fusion dual-modal CNN.

    Architecture
    ────────────
    SAR branch   (1-ch)  →  128×8×8  →  flatten  →  8192-dim
    Optical branch (3-ch) →  128×8×8  →  flatten  →  8192-dim
                                         concat   →  16384-dim
                                    fusion head   →  4 logits

    Fusion head: Linear(16384→128) + ReLU + Dropout(0.5) + Linear(128→4)

    Optional warm-start
    ───────────────────
    Call load_pretrained_branches() after instantiation to initialise both
    CNN branches from the single-modal checkpoints.  The fusion head is
    always trained from scratch so the branches can co-adapt.
    """

    def __init__(self, num_classes=4):
        super().__init__()
        self.sar_branch     = _build_feature_extractor(in_channels=1)
        self.optical_branch = _build_feature_extractor(in_channels=3)

        # Concatenated flat dim = 8192 + 8192 = 16384
        self.fusion_classifier = nn.Sequential(
            nn.Linear(FLAT_DIM * 2, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, sar_img, opt_img):
        sar_feat = self.sar_branch(sar_img)
        sar_feat = sar_feat.view(sar_feat.size(0), -1)   # (B, 8192)

        opt_feat = self.optical_branch(opt_img)
        opt_feat = opt_feat.view(opt_feat.size(0), -1)   # (B, 8192)

        fused = torch.cat([sar_feat, opt_feat], dim=1)   # (B, 16384)
        return self.fusion_classifier(fused)

    def load_pretrained_branches(self, sar_ckpt_path, opt_ckpt_path, device):
        """
        Warm-start both CNN branches from single-modal checkpoints.
        Only feature extractor weights are copied; fusion head stays random.
        """
        def _extract(state_dict, prefix="features."):
            return {k[len(prefix):]: v
                    for k, v in state_dict.items()
                    if k.startswith(prefix)}

        sar_ckpt = torch.load(sar_ckpt_path, map_location=device)
        opt_ckpt = torch.load(opt_ckpt_path, map_location=device)

        self.sar_branch.load_state_dict(_extract(sar_ckpt["model_state"]))
        self.optical_branch.load_state_dict(_extract(opt_ckpt["model_state"]))
        print("[Fusion_model] Pretrained branch weights loaded successfully.")

In [45]:
#utils.py
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import transforms


# ── Global config ──────────────────────────────────────────────────────────────
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 4
BATCH_SIZE  = 32
EPOCHS      = 20
LR          = 1e-4
SEED        = 42          # fixed seed → reproducible train/val/test splits
BASE_DIR    = (
    "/kaggle/input/datasets/requiemonk/"
    "sentinel12-image-pairs-segregated-by-terrain/v_2"
)
CLASS_NAMES = ["agri", "barrenland", "grassland", "urban"]

# ── Transforms ─────────────────────────────────────────────────────────────────
SAR_TRAIN_TRANSFORM = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

SAR_EVAL_TRANSFORM = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

OPTICAL_TRAIN_TRANSFORM = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

OPTICAL_EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# ── Internal split helper ──────────────────────────────────────────────────────
def _split_indices(total, seed=SEED):
    """
    Returns (train_idx, val_idx, test_idx) for a 70 / 15 / 15 split.
    Uses a fixed Generator so splits are identical across every run.
    """
    generator = torch.Generator().manual_seed(seed)
    indices   = torch.randperm(total, generator=generator).tolist()
    n_train   = int(0.70 * total)
    n_val     = int(0.15 * total)
    return (
        indices[:n_train],
        indices[n_train: n_train + n_val],
        indices[n_train + n_val:],
    )


# ── SAR dataloaders ────────────────────────────────────────────────────────────
def get_sar_dataloaders(base_dir=BASE_DIR, batch_size=BATCH_SIZE):
    train_ds = SAR_Dataset(base_dir, transform=SAR_TRAIN_TRANSFORM)
    val_ds   = SAR_Dataset(base_dir, transform=SAR_EVAL_TRANSFORM)
    test_ds  = SAR_Dataset(base_dir, transform=SAR_EVAL_TRANSFORM)

    train_idx, val_idx, test_idx = _split_indices(len(train_ds))

    train_loader = DataLoader(Subset(train_ds, train_idx),
                              batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(Subset(val_ds, val_idx),
                              batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(Subset(test_ds, test_idx),
                              batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader


# ── Optical dataloaders ────────────────────────────────────────────────────────
def get_optical_dataloaders(base_dir=BASE_DIR, batch_size=BATCH_SIZE):
    train_ds = Optical_Dataset(base_dir, transform=OPTICAL_TRAIN_TRANSFORM)
    val_ds   = Optical_Dataset(base_dir, transform=OPTICAL_EVAL_TRANSFORM)
    test_ds  = Optical_Dataset(base_dir, transform=OPTICAL_EVAL_TRANSFORM)

    train_idx, val_idx, test_idx = _split_indices(len(train_ds))

    train_loader = DataLoader(Subset(train_ds, train_idx),
                              batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(Subset(val_ds, val_idx),
                              batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(Subset(test_ds, test_idx),
                              batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader


# ── Fusion dataloaders ─────────────────────────────────────────────────────────
def get_fusion_dataloaders(base_dir=BASE_DIR, batch_size=BATCH_SIZE):
    """
    Three Fusion_Dataset instances share the same sorted pair order but
    receive different transforms (train augmentation vs eval-only).
    The same seeded randperm ensures the three splits are identical.
    """
    train_ds = Fusion_Dataset(base_dir,
                              sar_transform=SAR_TRAIN_TRANSFORM,
                              optical_transform=OPTICAL_TRAIN_TRANSFORM)
    val_ds   = Fusion_Dataset(base_dir,
                              sar_transform=SAR_EVAL_TRANSFORM,
                              optical_transform=OPTICAL_EVAL_TRANSFORM)
    test_ds  = Fusion_Dataset(base_dir,
                              sar_transform=SAR_EVAL_TRANSFORM,
                              optical_transform=OPTICAL_EVAL_TRANSFORM)

    train_idx, val_idx, test_idx = _split_indices(len(train_ds))

    train_loader = DataLoader(Subset(train_ds, train_idx),
                              batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(Subset(val_ds, val_idx),
                              batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(Subset(test_ds, test_idx),
                              batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader

In [46]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [47]:
#train.py
import torch
import torch.nn as nn
from tqdm import tqdm


def train_one_epoch(model, loader, criterion, optimizer, device, fusion=False):
    """
    Runs one full training epoch.

    Args:
        fusion: if True, expects batches of (sar_img, opt_img, label);
                if False, expects batches of (image, label).

    Returns:
        Average cross-entropy loss over the epoch.
    """
    model.train()
    total_loss = 0.0

    for batch in tqdm(loader, desc="  Train", leave=False):
        if fusion:
            sar_imgs, opt_imgs, labels = batch
            sar_imgs = sar_imgs.to(device)
            opt_imgs = opt_imgs.to(device)
            labels   = labels.to(device)
            optimizer.zero_grad()
            outputs  = model(sar_imgs, opt_imgs)
        else:
            images, labels = batch
            images  = images.to(device)
            labels  = labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, device, fusion=False):
    """
    Evaluates model accuracy on the given loader.

    Returns:
        Fraction of correctly classified samples (float in [0, 1]).
    """
    model.eval()
    correct = 0
    total   = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="  Eval ", leave=False):
            if fusion:
                sar_imgs, opt_imgs, labels = batch
                sar_imgs = sar_imgs.to(device)
                opt_imgs = opt_imgs.to(device)
                labels   = labels.to(device)
                outputs  = model(sar_imgs, opt_imgs)
            else:
                images, labels = batch
                images  = images.to(device)
                labels  = labels.to(device)
                outputs = model(images)

            preds    = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    return correct / total

In [50]:
#main.py
import os
import torch
import torch.nn as nn


# ── Checkpoint helpers ─────────────────────────────────────────────────────────

def _save_checkpoint(path, model, optimizer, epoch, best_val_acc):
    torch.save({
        "epoch":           epoch,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "best_val_acc":    best_val_acc,
    }, path)


def _load_checkpoint(path, model, optimizer, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    return ckpt["epoch"] + 1, ckpt["best_val_acc"]


# ── SAR training ───────────────────────────────────────────────────────────────

def sar_main():
    print("\n" + "="*60)
    print("  SAR-only training")
    print("="*60)

    ckpt_path = "best_model.pth"
    model     = SAR_model().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    start_epoch  = 0
    best_val_acc = 0.0

    if os.path.exists(ckpt_path):
        start_epoch, best_val_acc = _load_checkpoint(
            ckpt_path, model, optimizer, DEVICE)
        print(f"[SAR] Resumed from epoch {start_epoch} | "
              f"best_val_acc={best_val_acc:.4f}")

    train_loader, val_loader, test_loader = get_sar_dataloaders()

    for epoch in range(start_epoch, EPOCHS):
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, DEVICE)
        val_acc = evaluate(model, val_loader, DEVICE)

        print(f"[SAR] Epoch {epoch+1:02d}/{EPOCHS}  "
              f"loss={train_loss:.4f}  val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            _save_checkpoint(ckpt_path, model, optimizer, epoch, best_val_acc)
            print(f"       ✓ Checkpoint saved (best val_acc={best_val_acc:.4f})")

    test_acc = evaluate(model, test_loader, DEVICE)
    print(f"\n[SAR] Final test accuracy : {test_acc*100:.2f}%")
    return test_acc


# ── Optical training ───────────────────────────────────────────────────────────

def optical_main():
    print("\n" + "="*60)
    print("  Optical-only training")
    print("="*60)

    ckpt_path = "best_model_optical.pth"
    model     = Optical_model().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    start_epoch  = 0
    best_val_acc = 0.0

    if os.path.exists(ckpt_path):
        start_epoch, best_val_acc = _load_checkpoint(
            ckpt_path, model, optimizer, DEVICE)
        print(f"[Optical] Resumed from epoch {start_epoch} | "
              f"best_val_acc={best_val_acc:.4f}")

    train_loader, val_loader, test_loader = get_optical_dataloaders()

    for epoch in range(start_epoch, EPOCHS):
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, DEVICE)
        val_acc = evaluate(model, val_loader, DEVICE)

        print(f"[Optical] Epoch {epoch+1:02d}/{EPOCHS}  "
              f"loss={train_loss:.4f}  val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            _save_checkpoint(ckpt_path, model, optimizer, epoch, best_val_acc)
            print(f"          ✓ Checkpoint saved (best val_acc={best_val_acc:.4f})")

    test_acc = evaluate(model, test_loader, DEVICE)
    print(f"\n[Optical] Final test accuracy : {test_acc*100:.2f}%")
    return test_acc


# ── Fusion training ────────────────────────────────────────────────────────────

def fusion_main(pretrained=False):
    """
    Args:
        pretrained: if True, warm-start both CNN branches from the
                    single-modal checkpoints before training begins.
                    The fusion head is always randomly initialised.
    """
    print("\n" + "="*60)
    print("  Late-fusion training  (SAR + Optical)")
    print("="*60)

    ckpt_path = "best_model_fusion.pth"
    model     = Fusion_model().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    start_epoch  = 0
    best_val_acc = 0.0

    # Warm-start only if no fusion checkpoint exists yet
    if pretrained and not os.path.exists(ckpt_path):
        model.load_pretrained_branches(
            sar_ckpt_path="best_model.pth",
            opt_ckpt_path="best_model_optical.pth",
            device=DEVICE,
        )

    if os.path.exists(ckpt_path):
        start_epoch, best_val_acc = _load_checkpoint(
            ckpt_path, model, optimizer, DEVICE)
        print(f"[Fusion] Resumed from epoch {start_epoch} | "
              f"best_val_acc={best_val_acc:.4f}")

    train_loader, val_loader, test_loader = get_fusion_dataloaders()

    for epoch in range(start_epoch, EPOCHS):
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, DEVICE, fusion=True)
        val_acc = evaluate(model, val_loader, DEVICE, fusion=True)

        print(f"[Fusion] Epoch {epoch+1:02d}/{EPOCHS}  "
              f"loss={train_loss:.4f}  val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            _save_checkpoint(ckpt_path, model, optimizer, epoch, best_val_acc)
            print(f"         ✓ Checkpoint saved (best val_acc={best_val_acc:.4f})")

    test_acc = evaluate(model, test_loader, DEVICE, fusion=True)
    print(f"\n[Fusion] Final test accuracy : {test_acc*100:.2f}%")

    # ── Final comparison table ─────────────────────────────────────────────────
    print("\n" + "="*60)
    print("  RESULTS SUMMARY")
    print("="*60)
    print(f"  SAR only     : 98.38%")
    print(f"  Optical only : 98.08%")
    print(f"  Fusion        : {test_acc*100:.2f}%")
    print("="*60)
    return test_acc


# ── Entry point ────────────────────────────────────────────────────────────────

# if __name__ == "__main__":
    #sar_main()
    #optical_main()
    # fusion_main(pretrained=True)

In [53]:
"""
inference.py — Run predictions on new image pairs using saved checkpoints.

Usage examples
──────────────
# Single pair (fusion)
python inference.py \
    --mode fusion \
    --sar  path/to/image_s1.png \
    --opt  path/to/image_s2.png \
    --ckpt best_model_fusion.pth

# Single SAR image
python inference.py --mode sar --sar path/to/image.png --ckpt best_model.pth

# Single Optical image
python inference.py --mode optical --opt path/to/image.png --ckpt best_model_optical.pth

# Batch over a folder of matched pairs
python inference.py \
    --mode fusion \
    --sar_dir  /data/s1/ \
    --opt_dir  /data/s2/ \
    --ckpt best_model_fusion.pth
"""

import os
import argparse
import torch
import torch.nn.functional as F
from PIL import Image


# ── Model loader ───────────────────────────────────────────────────────────────

def load_model(mode, ckpt_path, device):
    if mode == "sar":
        model = SAR_model()
    elif mode == "optical":
        model = Optical_model()
    elif mode == "fusion":
        model = Fusion_model()
    else:
        raise ValueError(f"Unknown mode: {mode}")

    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    model.to(device).eval()
    print(f"[inference] Loaded {mode} checkpoint from '{ckpt_path}'  "
          f"(saved at epoch {ckpt['epoch']+1}, "
          f"best_val_acc={ckpt['best_val_acc']:.4f})")
    return model


# ── Single-image helpers ───────────────────────────────────────────────────────

def _load_sar(path):
    return SAR_EVAL_TRANSFORM(Image.open(path).convert("L")).unsqueeze(0)


def _load_opt(path):
    return OPTICAL_EVAL_TRANSFORM(Image.open(path).convert("RGB")).unsqueeze(0)


# ── Predict functions ──────────────────────────────────────────────────────────

def predict_sar(model, sar_path, device):
    img = _load_sar(sar_path).to(device)
    with torch.no_grad():
        logits = model(img)
    probs = F.softmax(logits, dim=1).squeeze()
    pred  = probs.argmax().item()
    return {
        "predicted_class": CLASS_NAMES[pred],
        "confidence":      round(probs[pred].item(), 4),
        "probabilities":   {cls: round(probs[i].item(), 4)
                            for i, cls in enumerate(CLASS_NAMES)},
    }


def predict_optical(model, opt_path, device):
    img = _load_opt(opt_path).to(device)
    with torch.no_grad():
        logits = model(img)
    probs = F.softmax(logits, dim=1).squeeze()
    pred  = probs.argmax().item()
    return {
        "predicted_class": CLASS_NAMES[pred],
        "confidence":      round(probs[pred].item(), 4),
        "probabilities":   {cls: round(probs[i].item(), 4)
                            for i, cls in enumerate(CLASS_NAMES)},
    }


def predict_fusion(model, sar_path, opt_path, device):
    sar = _load_sar(sar_path).to(device)
    opt = _load_opt(opt_path).to(device)
    with torch.no_grad():
        logits = model(sar, opt)
    probs = F.softmax(logits, dim=1).squeeze()
    pred  = probs.argmax().item()
    return {
        "predicted_class": CLASS_NAMES[pred],
        "confidence":      round(probs[pred].item(), 4),
        "probabilities":   {cls: round(probs[i].item(), 4)
                            for i, cls in enumerate(CLASS_NAMES)},
    }


# ── Batch over a folder ────────────────────────────────────────────────────────

def batch_predict_fusion(model, sar_dir, opt_dir, device):
    """
    Iterates all .png files in sar_dir, derives paired optical filename
    (_s1_ → _s2_), and prints predictions for each matched pair.
    """
    results = []
    missing = 0
    files   = sorted(f for f in os.listdir(sar_dir) if f.endswith(".png"))

    print(f"\n[batch] Processing {len(files)} SAR files from '{sar_dir}'")
    for fname in files:
        opt_fname = fname.replace("_s1_", "_s2_")
        sar_path  = os.path.join(sar_dir, fname)
        opt_path  = os.path.join(opt_dir, opt_fname)

        if not os.path.exists(opt_path):
            print(f"  [SKIP] No optical pair for {fname}")
            missing += 1
            continue

        result = predict_fusion(model, sar_path, opt_path, device)
        result["sar_file"] = fname
        results.append(result)
        print(f"  {fname}  →  {result['predicted_class']:<12} "
              f"(conf={result['confidence']:.4f})")

    print(f"\n[batch] Done. Predicted: {len(results)}  |  Skipped: {missing}")
    return results


# ── CLI ────────────────────────────────────────────────────────────────────────

def _parse_args():
    p = argparse.ArgumentParser(
        description="SAR / Optical / Fusion land-cover inference")
    p.add_argument("--mode",    required=True,
                   choices=["sar", "optical", "fusion"],
                   help="Which model to use")
    p.add_argument("--ckpt",    required=True,
                   help="Path to checkpoint .pth file")
    p.add_argument("--sar",     default=None,
                   help="Path to a single SAR image (.png)")
    p.add_argument("--opt",     default=None,
                   help="Path to a single Optical image (.png)")
    p.add_argument("--sar_dir", default=None,
                   help="Folder of SAR images for batch prediction (fusion only)")
    p.add_argument("--opt_dir", default=None,
                   help="Folder of Optical images for batch prediction (fusion only)")
    return p.parse_args()


def main():
    args  = _parse_args()
    model = load_model(args.mode, args.ckpt, DEVICE)

    # ── Batch mode ─────────────────────────────────────────────────────────────
    if args.sar_dir and args.opt_dir:
        if args.mode != "fusion":
            raise ValueError("--sar_dir/--opt_dir batch mode only supports --mode fusion")
        batch_predict_fusion(model, args.sar_dir, args.opt_dir, DEVICE)
        return

    # ── Single-image mode ──────────────────────────────────────────────────────
    if args.mode == "sar":
        if not args.sar:
            raise ValueError("--sar is required for mode=sar")
        result = predict_sar(model, args.sar, DEVICE)

    elif args.mode == "optical":
        if not args.opt:
            raise ValueError("--opt is required for mode=optical")
        result = predict_optical(model, args.opt, DEVICE)

    elif args.mode == "fusion":
        if not args.sar or not args.opt:
            raise ValueError("--sar and --opt are both required for mode=fusion")
        result = predict_fusion(model, args.sar, args.opt, DEVICE)

    print(f"\nPredicted class : {result['predicted_class']}")
    print(f"Confidence      : {result['confidence']:.4f}")
    print("Class probabilities:")
    for cls, prob in result["probabilities"].items():
        bar = "█" * int(prob * 40)
        print(f"  {cls:<12} {prob:.4f}  {bar}")


if __name__ == "__main__":
    main()

usage: colab_kernel_launcher.py [-h] --mode {sar,optical,fusion} --ckpt CKPT
                                [--sar SAR] [--opt OPT] [--sar_dir SAR_DIR]
                                [--opt_dir OPT_DIR]
colab_kernel_launcher.py: error: the following arguments are required: --mode, --ckpt
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.12/argparse.py", line 1943, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2230, in _parse_known_args
    raise ArgumentError(None, _('the following arguments are required: %s') %
argparse.ArgumentError: the following arguments are required: --mode, --ckpt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_55/3860714846.py", line 173, in <cell line: 0>
    args = p.parse_args()
           ^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 1904, in parse_args
    args, argv = self.parse_known_args(args, namespace)
   

TypeError: object of type 'NoneType' has no len()